In [4]:
#!pip install streamlit streamlit-folium

  Using cached blinker-1.8.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached pyarrow-17.0.0-cp38-cp38-win_amd64.whl.metadata (3.4 kB)
  Using cached tenacity-9.0.0-py3-none-any.whl.metadata (1.2 kB)
   ---------------------------------------- 0.0/8.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.6 MB 1.1 MB/s eta 0:00:08
   -- ------------------------------------- 0.5/8.6 MB 1.1 MB/s eta 0:00:08
   --- ------------------------------------ 0.8/8.6 MB 907.1 kB/s eta 0:00:09
   ---- ----------------------------------- 1.0/8.6 MB 931.8 kB/s eta 0:00:09
   ---- ----------------------------------- 1.0/8.6 MB 931.8 kB/s eta 0:00:09
   ---- ----------------------------------- 1.0/8.6 MB 931.8 kB/s eta 0:00:09
   ------- -------------------------------- 1.6/8.6 MB 902.

In [1]:
import sys
import os
sys.path.append(os.path.abspath("../"))

Pour identifier les services les plus proches, nous utilisons la distance de Haversine, adaptée aux coordonnées géographiques qui contrairement à une distance euclidienne tient compte de la courbure de la Terre.


 $$
d = 2R \arcsin \left( \sqrt{ \sin^2\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta \lambda}{2}\right)} \right)
$$

où :

- $\phi$ : latitude
- $\lambda$ : longitude
- $R$ : rayon de la Terre


Les distances sont ensuite calculées pour chaque point et triées afin de sélectionner les plus proches.

In [2]:
from src.nearest_neighbors import find_nearest
import pandas as pd

df = pd.read_csv("../data/pos_cotonou.csv")

# position test (ex : Cotonou centre)
user_lat = 6.37
user_lon = 2.43

nearest_restaurants = find_nearest(
    df,
    user_lat,
    user_lon,
    category="restaurant",
    top_n=10
)

nearest_restaurants

,city,lat,lon,category,name,source,distance_km
338,Cotonou,6.371323,2.427967,restaurant,Milles et une Bouillie,OSM,0.268546
295,Cotonou,6.367488,2.429032,restaurant,Les Grillades d'Ayinkè,OSM,0.299086
222,Cotonou,6.366712,2.429727,restaurant,Haikuo tiankong,OSM,0.366852
381,Cotonou,6.365047,2.427991,restaurant,Lion Indomptable,OSM,0.593837
223,Cotonou,6.364607,2.429781,restaurant,Maquis Dose dit Diallo,OSM,0.600186
396,Cotonou,6.364766,2.428310,restaurant,Sénégalais Tchep,OSM,0.611240
226,Cotonou,6.364198,2.430086,restaurant,Maquis Bar Chez tata vivi,OSM,0.645268
227,Cotonou,6.364401,2.431584,restaurant,Maquis Bar Djembè,OSM,0.646725
225,Cotonou,6.363998,2.429825,restaurant,Maquis Chez Angele,OSM,0.667706
219,Cotonou,6.364352,2.432302,restaurant,Bombay Tandoor,OSM,0.677587


# Visualisation carte

In [3]:
import folium

m = folium.Map(location=[user_lat, user_lon], zoom_start=14)

# utilisateur
folium.Marker(
    [user_lat, user_lon],
    popup="Utilisateur",
    icon=folium.Icon(color="red")
).add_to(m)

# points proches
for _, row in nearest_restaurants.iterrows():
    folium.Marker(
        [row["lat"], row["lon"]],
        popup=f"{row['name']} ({row['distance_km']:.2f} km)"
    ).add_to(m)

m